<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z330_RegresionLinealReciente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regresión Lineal Reciente — Local Linear Trend

## La idea

Mirando las series de los productos con quiebre estructural se nota algo: en muchos casos la caída **ya estaba en curso** en los últimos 6 meses de historia. El HAR, AutoGluon y el naive la ignoran porque están anclados al nivel histórico largo.

La hipótesis es que una **regresión lineal sobre los últimos N meses** captura esa tendencia reciente y predice más bajo para los productos en caída, sin necesitar aprender nada.

```
HAR / AutoGluon: usan historia larga → anclados al nivel histórico
RegLineal 6m:    tn = a + b·t  (solo últimos 6 meses) → sigue la tendencia reciente
```

Para productos estables (b ≈ 0) da igual que el naive.
Para productos en caída (b < 0) predice más bajo → error menor.
Para productos en subida (b > 0) predice más alto — riesgo de sobre-predecir.

## Variantes

| Variante | Ventana | Intuición |
|---|---|---|
| `reg3m` | 3 meses | Tendencia muy reciente, más reactiva |
| `reg6m` | 6 meses | Balance entre señal reciente y estabilidad |
| `reg12m` | 12 meses | Tendencia anual completa |
| `reg6m_piso` | 6 meses | Con piso en mediana 3m — no predice por debajo del mínimo reciente |

## Backtesting interno

Comparamos contra Naive y HAR sobre el mismo split: train hasta 201910, target 201912.

## 0.1 Init ambiente Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Setup

In [ ]:
!pip install uv
!uv pip install -q kaggle

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
  import os
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  os.system(comando)

In [ ]:
import os
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from scipy.stats import chi2, runstest_1samp
from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings('ignore')

In [ ]:
PARAM = {
  'experimento':        'RegLinealReciente-01',
  'kaggle_competition': 'labo-iii-2026-rosario',
  'semilla_primigenia': 102191,
  # ventana para el submit final (opciones: 3, 6, 12, 'piso')
  'variante_submit':    6,
  # backtesting
  'periodo_corte':  201910,
  'periodo_target': 201912,
}

In [ ]:
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

# 2  Datos

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")
tb_ventas    = tb_ventas.join(tb_apredecir, on="product_id", how="inner").sort(["product_id", "periodo"])

tb_train = tb_ventas.filter(pl.col("periodo") <= PARAM['periodo_corte'])
tb_real  = (
    tb_ventas
    .filter(pl.col("periodo") == PARAM['periodo_target'])
    .select(["product_id", "tn"])
    .rename({"tn": "tn_real"})
)

productos = tb_apredecir["product_id"].to_list()
print(f"{len(productos)} productos")

# 3  Función de predicción — regresión lineal reciente

Ajusta `tn = a + b·t` sobre los últimos `ventana` meses y extrapola a t+2.

Con `piso=True` aplica un piso en la mediana de los últimos 3 meses — evita que la extrapolación llegue a valores negativos o absurdamente bajos en series con tendencia muy pronunciada.

In [ ]:
def pred_reg_lineal(serie: np.ndarray, ventana: int, horizonte: int = 2,
                    piso: bool = False) -> float:
    """
    Ajusta recta sobre los últimos `ventana` puntos y predice `horizonte` pasos adelante.
    """
    n = len(serie)
    w = min(ventana, n)  # no pedir más de lo que hay

    if w < 2:
        return max(float(serie.mean()), 0.0)

    y = serie[-w:]
    x = np.arange(w).reshape(-1, 1)

    m = LinearRegression().fit(x, y)
    pred = float(m.predict([[w - 1 + horizonte]])[0])

    if piso:
        # piso: mediana de los últimos 3 meses (no predecir por debajo del nivel más reciente)
        piso_val = float(np.median(serie[-3:]))
        pred = max(pred, piso_val * 0.5)  # permite caer hasta el 50% del mínimo reciente

    return max(pred, 0.0)

# 4  Backtesting — comparación de variantes

Train hasta 201910, target 201912. Calculamos todas las variantes y el Naive + HAR de referencia.

In [ ]:
# helpers HAR
def build_har_features(serie):
    T = len(serie)
    X, y = [], []
    for t in range(12, T):
        X.append([serie[t-1], serie[t-3:t].mean(), serie[t-6:t].mean(), serie[t-12:t].mean()])
        y.append(serie[t])
    return np.array(X), np.array(y)

def har_predict_next(m, serie):
    t = len(serie)
    x = np.array([[serie[t-1], serie[t-3:t].mean(), serie[t-6:t].mean(), serie[t-12:t].mean()]])
    return float(m.predict(x)[0])


resultados = []

for pid in productos:
    serie = (
        tb_train.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )

    # naive
    w6    = serie[max(0, len(serie)-6):]
    naive = max(float(np.median(w6)), 0.0)

    # HAR + Racha
    fallback = float(serie[-12:].mean()) if len(serie) >= 12 else float(serie.mean())
    try:
        _, pv = runstest_1samp(serie, cutoff='median')
        tiene_estructura = pv < 0.05
    except Exception:
        tiene_estructura = False

    if not tiene_estructura or len(serie) < 14:
        har = fallback
    else:
        try:
            X, y = build_har_features(serie)
            mhar = LinearRegression().fit(X, y)
            p1   = max(har_predict_next(mhar, serie), 0.0)
            har  = max(har_predict_next(mhar, np.append(serie, p1)), 0.0)
        except Exception:
            har = fallback

    # variantes de regresión lineal reciente
    reg3m      = pred_reg_lineal(serie, ventana=3,  horizonte=2)
    reg6m      = pred_reg_lineal(serie, ventana=6,  horizonte=2)
    reg12m     = pred_reg_lineal(serie, ventana=12, horizonte=2)
    reg6m_piso = pred_reg_lineal(serie, ventana=6,  horizonte=2, piso=True)

    resultados.append({
        'product_id': pid,
        'pred_naive':      naive,
        'pred_har':        har,
        'pred_reg3m':      reg3m,
        'pred_reg6m':      reg6m,
        'pred_reg12m':     reg12m,
        'pred_reg6m_piso': reg6m_piso,
    })

tb_preds = pl.DataFrame(resultados)
print("Predicciones listas")

In [ ]:
tb_bt = tb_real.join(tb_preds, on='product_id', how='left')

modelos_bt = ['naive', 'har', 'reg3m', 'reg6m', 'reg12m', 'reg6m_piso']
for m in modelos_bt:
    tb_bt = tb_bt.with_columns(
        (pl.col('tn_real') - pl.col(f'pred_{m}')).abs().alias(f'err_{m}')
    )

print("RMSE por modelo (backtesting 201912):")
print()
rmse_dict = {}
for m in modelos_bt:
    rmse = float(np.sqrt((tb_bt[f'err_{m}'] ** 2).mean()))
    rmse_dict[m] = rmse
    delta = rmse - rmse_dict['naive'] if m != 'naive' else 0
    signo = f'({delta:+.4f} vs naive)' if m != 'naive' else ''
    print(f"  {m:15s}: {rmse:.4f}  {signo}")

# 5  ¿Qué pasa en los productos con caída?

Separamos los productos según la tendencia de los últimos 6 meses.
La hipótesis es que reg6m debería ganar en los que ya venían cayendo.

In [ ]:
# calculamos pendiente normalizada para cada producto
pendientes = []
for pid in productos:
    serie = (
        tb_train.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )
    if len(serie) >= 6:
        ult6 = serie[-6:]
        slope = float(np.polyfit(np.arange(6), ult6, 1)[0])
        slope_norm = slope / (serie.mean() + 1e-9)
    else:
        slope_norm = 0.0
    pendientes.append({'product_id': pid, 'pendiente_norm': slope_norm})

tb_bt = tb_bt.join(pl.DataFrame(pendientes), on='product_id', how='left')

# grupos
tb_caida   = tb_bt.filter(pl.col('pendiente_norm') < -0.05)
tb_estable = tb_bt.filter((pl.col('pendiente_norm') >= -0.05) & (pl.col('pendiente_norm') <= 0.05))
tb_subida  = tb_bt.filter(pl.col('pendiente_norm') > 0.05)

print(f"Productos en caída   (pendiente < -5%/mes): {tb_caida.height}")
print(f"Productos estables   (-5% a +5%):           {tb_estable.height}")
print(f"Productos en subida  (pendiente > +5%/mes): {tb_subida.height}")

print()
for nombre, grupo in [('EN CAÍDA', tb_caida), ('ESTABLES', tb_estable), ('EN SUBIDA', tb_subida)]:
    print(f"--- {nombre} ---")
    for m in modelos_bt:
        rmse = float(np.sqrt((grupo[f'err_{m}'] ** 2).mean()))
        print(f"  {m:15s}: {rmse:.4f}")
    print()

# 6  Visualización: reg6m vs naive por grupo

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
grupos    = [('EN CAÍDA', tb_caida), ('ESTABLES', tb_estable), ('EN SUBIDA', tb_subida)]
colores   = ['tomato', 'steelblue', 'seagreen']

for i, (nombre, grupo) in enumerate(grupos):
    ea = np.log1p(grupo['err_naive'].to_numpy())
    eb = np.log1p(grupo['err_reg6m'].to_numpy())

    # reg6m gana cuando eb < ea (puntos por encima de la diagonal)
    reg_gana  = eb < ea
    naive_gana = ea <= eb

    axes[i].scatter(ea[naive_gana], eb[naive_gana], c='lightgray', s=15, alpha=0.6, label='naive gana')
    axes[i].scatter(ea[reg_gana],   eb[reg_gana],   c=colores[i],  s=15, alpha=0.7, label='reg6m gana')

    lim = max(ea.max(), eb.max()) * 1.05
    axes[i].plot([0, lim], [0, lim], 'k--', linewidth=0.8)
    axes[i].set_xlabel('log(1 + err_naive)', fontsize=9)
    axes[i].set_ylabel('log(1 + err_reg6m)', fontsize=9)
    axes[i].set_title(
        f'{nombre} (n={len(ea)})\nreg6m gana en {reg_gana.sum()} de {len(ea)}',
        fontsize=9
    )
    axes[i].legend(fontsize=7)

fig.suptitle('Naive vs reg6m por grupo de tendencia reciente\n(puntos sobre la diagonal = reg6m mejor)', fontsize=10)
plt.tight_layout()
plt.show()

# 7  McNemar: ¿reg6m supera al naive significativamente?

In [ ]:
def mcnemar(err_a, err_b, nombre_a, nombre_b):
    n10 = (err_a < err_b).sum()
    n01 = (err_b < err_a).sum()
    if n10 + n01 == 0:
        print(f"{nombre_a} vs {nombre_b}: sin discrepancias")
        return
    chi2_stat = (abs(n10 - n01) - 1)**2 / (n10 + n01)
    pvalue = 1 - chi2.cdf(chi2_stat, df=1)
    ganador = nombre_a if n10 > n01 else nombre_b
    sig = "** SIGNIFICATIVO **" if pvalue < 0.05 else "no significativo"
    print(f"{nombre_a:15s} vs {nombre_b:15s}:  {nombre_a} gana {n10:3d} | {nombre_b} gana {n01:3d}  p={pvalue:.4f}  {sig}  → {ganador}")

print("McNemar global (780 productos):")
ea_n = tb_bt['err_naive'].to_numpy()
for m in ['reg3m', 'reg6m', 'reg12m', 'reg6m_piso', 'har']:
    mcnemar(ea_n, tb_bt[f'err_{m}'].to_numpy(), 'naive', m)

print()
print("McNemar solo en productos EN CAÍDA:")
ea_c = tb_caida['err_naive'].to_numpy()
for m in ['reg3m', 'reg6m', 'reg12m', 'reg6m_piso', 'har']:
    mcnemar(ea_c, tb_caida[f'err_{m}'].to_numpy(), 'naive', m)

# 8  Visualización de casos donde reg6m gana

Series donde reg6m superó claramente al naive — confirmamos visualmente que la tendencia reciente era real.

In [ ]:
# productos donde reg6m ganó más al naive
tb_bt = tb_bt.with_columns(
    (pl.col('err_naive') - pl.col('err_reg6m')).alias('mejora_reg6m')
)

top_mejora = (
    tb_bt.filter(pl.col('pendiente_norm') < -0.05)
    .sort('mejora_reg6m', descending=True)
    .head(6)
    ['product_id'].to_list()
)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, pid in enumerate(top_mejora):
    serie_full = tb_ventas.filter(pl.col('product_id') == pid).sort('periodo')
    periodos_  = serie_full['periodo'].to_list()
    tn_        = serie_full['tn'].to_numpy().astype(float)

    idx_corte  = next((j for j, p in enumerate(periodos_) if p > PARAM['periodo_corte']), len(periodos_))
    idx_target = next((j for j, p in enumerate(periodos_) if p == PARAM['periodo_target']), None)
    tn_train_  = tn_[:idx_corte]

    # línea de regresión sobre últimos 6 meses
    ult6_  = tn_train_[-6:]
    x6_    = np.arange(len(tn_train_) - 6, len(tn_train_))
    coef_  = np.polyfit(x6_, ult6_, 1)

    row = tb_bt.filter(pl.col('product_id') == pid)
    real_val  = float(row['tn_real'][0])
    naive_val = float(row['pred_naive'][0])
    reg6_val  = float(row['pred_reg6m'][0])
    har_val   = float(row['pred_har'][0])
    mejora    = float(row['mejora_reg6m'][0])

    ax = axes[i]
    ax.plot(range(len(tn_train_)), tn_train_, 'o-', color='steelblue',
            markersize=3, linewidth=1.5, label='historia')

    # extensión de la recta hasta t+2
    x_ext = np.array([len(tn_train_) - 6, len(tn_train_) + 1])
    ax.plot(x_ext, np.polyval(coef_, x_ext), 'r-', linewidth=2,
            alpha=0.85, label='reg lineal 6m')

    if idx_target is not None:
        ax.scatter([idx_target], [real_val],  color='black',  s=90,  zorder=6, label=f'real={real_val:.1f}')
        ax.scatter([idx_target], [reg6_val],  color='tomato', s=60,  zorder=5, marker='D', label=f'reg6m={reg6_val:.1f}')
        ax.scatter([idx_target], [naive_val], color='gray',   s=40,  zorder=5, marker='D', label=f'naive={naive_val:.1f}')
        ax.scatter([idx_target], [har_val],   color='green',  s=40,  zorder=5, marker='D', label=f'HAR={har_val:.1f}')

    ax.set_title(f'pid {pid}  |  mejora reg6m={mejora:.1f}', fontsize=8)
    ax.legend(fontsize=5.5)

fig.suptitle('Casos donde reg6m gana claramente al naive\n(recta roja = tendencia 6m extrapolada)', fontsize=10)
plt.tight_layout()
plt.show()

# 9  Predicción final para 202002 y submit

Usamos toda la historia hasta 201912 para predecir 202002 (t+2).
Cambiá `PARAM['variante_submit']` para probar distintas ventanas.

In [ ]:
# usamos todos los datos hasta 201912 para el submit final
tb_full = tb_ventas  # ya filtrado a los 780 productos

preds_final = []
for pid in productos:
    serie = (
        tb_full.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )

    v = PARAM['variante_submit']
    if v == 'piso':
        pred = pred_reg_lineal(serie, ventana=6, horizonte=2, piso=True)
    else:
        pred = pred_reg_lineal(serie, ventana=int(v), horizonte=2)

    preds_final.append({'product_id': pid, 'tn': pred})

tb_final = pl.DataFrame(preds_final)
display(tb_final.head(10))
print(f"Nulls: {tb_final['tn'].is_null().sum()}")

In [ ]:
v = PARAM['variante_submit']
archivo = f"RegLineal_rec{v}m.csv"
mensaje = f"Regresion lineal ultimos {v} meses → t+2"

tb_final.write_csv(archivo)
kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje)

# 10  Qué probar

| Cambio | Dónde | Por qué |
|---|---|---|
| `'variante_submit': 3` | PARAM | Más reactivo, sigue los últimos 3 meses |
| `'variante_submit': 'piso'` | PARAM | 6 meses con piso — evita predicciones absurdamente bajas |
| `'variante_submit': 12` | PARAM | Tendencia anual — más estable, menos reactiva |
| Combinar con naive: `0.5*reg6m + 0.5*naive` | celda nueva | Promedia el ancla histórica con la tendencia reciente |
| Usar reg6m solo en productos con pendiente < -5% | celda nueva | Aplica solo donde tiene sentido, naive para el resto |